In [ ]:
# Install required packages (pycocotools, opencv, wget)
# Then download the COCO 2017 validation images and annotations.
!pip install pycocotools opencv-python wget -q

import os
import wget
import zipfile

# Create a temporary directory for COCO data
os.makedirs("/content/coco", exist_ok=True)

# Download COCO 2017 validation annotations
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P /content/coco/
!unzip -q /content/coco/annotations_trainval2017.zip -d /content/coco/

# Download COCO 2017 validation images
!wget -q http://images.cocodataset.org/zips/val2017.zip -P /content/coco/
!unzip -q /content/coco/val2017.zip -d /content/coco/

print("COCO dataset ready.")

# Synthetic Anomaly Generation via Copy‑Paste from COCO

This script creates a synthetic anomaly detection dataset by pasting objects from COCO (excluding classes that already appear in Cityscapes) onto random Cityscapes background images. The output is a set of images and binary masks (anomaly = 255) saved under `output_dir`.

In [ ]:

import zipfile
import random
import cv2
import numpy as np
from PIL import Image
from io import BytesIO
from pycocotools.coco import COCO

# Paths
# Cityscapes ZIP file containing all images (must be already uploaded to Drive)
cityscapes_img_zip = "/content/drive/MyDrive/ML/dataset/leftImg8bit_trainvaltest.zip" #TODO: change path

# Output directory where generated images and masks will be saved 
output_dir = "/content/drive/MyDrive/ML/dataset/anomaly_training" #TODO: change path

# Cityscapes split to use as background: 'train' (2975 images) or 'val' (500)
subset = "train"

# Number of synthetic images to generate
num_images_to_generate = 1000

# COCO paths 
coco_ann_path = "/content/coco/annotations/instances_val2017.json"
coco_img_dir = "/content/coco/val2017"
coco = COCO(coco_ann_path)


# Define which COCO categories to EXCLUDE (they are already present in
# Cityscapes or are too similar – we want only unusual objects as anomalies).
exclude_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck',
    'traffic light', 'stop sign', 'parking meter'   # traffic signs in general
]

# Map category names to COCO ids
all_cats = coco.loadCats(coco.getCatIds())
exclude_ids = set()
for cat in all_cats:
    if cat['name'] in exclude_names:
        exclude_ids.add(cat['id'])

# List of category ids that are allowed (i.e., anomalies)
allowed_cat_ids = [cid for cid in coco.getCatIds() if cid not in exclude_ids]

print(f"Allowed COCO categories: {len(allowed_cat_ids)} out of {len(all_cats)} total")
print("Example allowed categories:", [cat['name'] for cat in coco.loadCats(allowed_cat_ids[:5])])

#####################################################################
# Read Cityscapes ZIP file and list all images for the chosen split
img_zip = zipfile.ZipFile(cityscapes_img_zip, 'r')
img_prefix = f"leftImg8bit/{subset}/"
cityscapes_files = [
    name for name in img_zip.namelist()
    if name.startswith(img_prefix) and name.endswith('_leftImg8bit.png')
]
print(f"Cityscapes {subset} images available: {len(cityscapes_files)}")

# Create output directories for images and masks
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "masks"), exist_ok=True)

# Core copy‑paste function
def paste_coco_object(cityscape_img, coco, allowed_cat_ids):
    # Randomly selects a COCO object from an allowed category,
    # extracts it, scales it, and pastes it onto a copy of the cityscape image.
    # At the end Returns: (modified_image, binary_mask, success_flag)
    
    # Pick a random COCO image
    img_id = random.choice(coco.getImgIds())
    # Get annotations for that image that belong to allowed categories and are not crowd
    ann_ids = coco.getAnnIds(imgIds=img_id, catIds=allowed_cat_ids, iscrowd=False)
    if len(ann_ids) == 0:
        return None, None, False

    ann_id = random.choice(ann_ids)
    ann = coco.loadAnns(ann_id)[0]
    if 'segmentation' not in ann or not ann['segmentation']:
        return None, None, False

    # Load the COCO image
    img_info = coco.loadImgs(img_id)[0]
    coco_path = os.path.join(coco_img_dir, img_info['file_name'])
    coco_img = cv2.imread(coco_path)
    if coco_img is None:
        return None, None, False
    coco_img = cv2.cvtColor(coco_img, cv2.COLOR_BGR2RGB)

    # Generate binary mask for the selected object
    mask = coco.annToMask(ann)
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None, None, False

    # Crop the object using its bounding box
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    object_img = coco_img[y_min:y_max+1, x_min:x_max+1].copy()
    object_mask = mask[y_min:y_max+1, x_min:x_max+1].astype(np.uint8)

    # Random scaling factor (0.3 to 1.0)
    scale = random.uniform(0.3, 1.0)
    new_h = int(object_img.shape[0] * scale)
    new_w = int(object_img.shape[1] * scale)
    if new_h < 1 or new_w < 1:
        return None, None, False

    # Resize both the object image and its mask
    object_img = cv2.resize(object_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    object_mask = cv2.resize(object_mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # Random paste position on the Cityscapes image
    city_h, city_w = cityscape_img.shape[:2]
    if new_h >= city_h or new_w >= city_w:
        return None, None, False

    x_offset = random.randint(0, city_w - new_w)
    y_offset = random.randint(0, city_h - new_h)

    # Alpha blending: to paste the object onto the background
    roi = cityscape_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w].astype(np.float32)
    object_rgb = object_img.astype(np.float32)
    alpha = (object_mask > 0).astype(np.float32)[:, :, np.newaxis]
    blended = (1 - alpha) * roi + alpha * object_rgb
    cityscape_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = blended.astype(np.uint8)

    # Create the full binary mask for this image (0 = background, 255 = anomaly)
    full_mask = np.zeros((city_h, city_w), dtype=np.uint8)
    full_mask[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = object_mask * 255

    return cityscape_img, full_mask, True


# Generation loop
generated = 0
while generated < num_images_to_generate:
    # Randomly pick a Cityscapes image from the ZIP
    img_name = random.choice(cityscapes_files)
    img_bytes = img_zip.read(img_name)
    cityscape_original = np.array(Image.open(BytesIO(img_bytes)).convert('RGB'))

    # Try up to 20 times to paste a valid COCO object onto this background
    success = False
    for _ in range(20):
        result_img, result_mask, ok = paste_coco_object(cityscape_original.copy(), coco, allowed_cat_ids)
        if ok:
            # Save the generated image and mask
            img_save_name = f"anomaly_{generated:04d}.png"
            mask_save_name = f"mask_{generated:04d}.png"
            Image.fromarray(result_img).save(os.path.join(output_dir, "images", img_save_name))
            Image.fromarray(result_mask).save(os.path.join(output_dir, "masks", mask_save_name))
            generated += 1
            if generated % 100 == 0:
                print(f"Generated {generated}/{num_images_to_generate}")
            success = True
            break
    if not success:
        # If no valid object could be pasted onto this background, skip it and try another background
        continue

print("Generation completed!")
img_zip.close()